In [0]:
%sql
create table if not exists customers_silver
(customer_id String,email string,first_name string, last_name string, gender string, street string, city string, country string)

In [0]:
df_country_lookup = spark.read.json("/Volumes/workspace/bookstore_eng_pro/dataset/country_lookup/")


In [0]:
from pyspark.sql import functions as F

customer_schema = "customer_id String,email string,first_name string, last_name string, gender string, street string, city string, country_code string, row_status string, row_time timestamp"

customers_df = (spark.read.table("workspace.bookstore_eng_pro.bronze")
                          .where("topic = 'customers'")
                          .select(F.from_json(F.unbase64(F.col("value")).cast("string"), customer_schema).alias("v"))
                          .select("v.*")
                          .filter(F.col("row_status").isin(["insert", "update"]))
)

display(customers_df)

In [0]:
from pyspark.sql.window import Window

window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

ranked_df = (customers_df.withColumn("rank", F.rank().over(window))
             .filter(F.col("rank") == 1)
             .drop("rank"))
display(ranked_df)

In [0]:
from pyspark.sql.window import Window

def batch_upsert(microBatchDF, batch):
    window = Window.partitionBy("customer_id").orderBy(F.col("row_time").desc())

    (microBatchDF.filter(F.col("row_status").isin(["insert","update"]))
                  .withColumn("rank", F.rank().over(window))
                  .filter(F.col("rank")==1)
                  .drop("rank")
                  .createOrReplaceTempView("ranked_updates"))
    
    sql_query = """
    MERGE INTO customers_silver a
    using ranked_updates b 
    on a.customer_id = b.customer_id
    WHEN MATCHED and a.row_time < b.row_time then 
    update set *
    when not matched then 
    insert *
    """

    microBatchDF.sparkSession.sql(sql_query)

In [0]:
%sql
select cast(unbase64(value) as string) from workspace.bookstore_eng_pro.bronze
where topic = "customers"